#### Preparation

- Data link: https://github.com/sakura-lee25/Data-Science-for-Spatial-Systems_final-assessment/tree/main/data
  
- Number of words: ***

- Runtime: 31 seconds

- Coding environment: SDS Docker (`jreades/sds:2025-arm` via Podman)

- License: this notebook is made available under the [Creative Commons Attribution license](https://creativecommons.org/licenses/by/4.0/).

- Additional libraries *[libraries not included in SDS Docker or not used in this module]*:
    - **loguru 0.7.3**: Advanced logging library with colour-coded output.
    - **mgwr** 2.2.1: Multi-scale Geographically Weighted Regression.
    - **imbalanced-learn 0.12.4** (>=0.11.0; version depends on Python): SMOTE oversampling for imbalanced classification.
    - **watermark 2.6.0**: A Jupyter Notebook extension for printing timestamps, version numbers, and hardware information.
    - **esda 2.5.1**: Exploratory Spatial Data Analysis (Moran's I, LISA)
  

# Spatial Analysis of Road Traffic Accident Risk in England: A Multi-scale Approach Using MGWR and Machine Learning

## Table of contents

1. [Introduction](#1.0-Introduction)

1. [Literature Reviews](#2.0-Literature-Review)

1. [Methodology](#3.0-Methodology)

1. [Data](#4.0-Data)

1. [Analysis](#5.0-Analysis)

1. [Discussion and Conclusion](#6.0-Discussion-and-Conclusion)

1. [References](#References)

<a id="Introduction"></a>
## 1.0 Introduction ##


Road traffic accidents remain a critical public health and urban governance challenge in the United Kingdom, causing approximately 1,700 fatalities and over 130,000 casualties annually (Department for Transport, 2024). While national casualty figures have declined over the past decade, this improvement has not been distributed evenly: the most deprived areas of England record pedestrian casualty rates nearly three times those of the least deprived, a disparity that has persisted throughout 2020-2024 (Department for Transport, 2025). Translating raw collision counts into population-normalised rates is therefore not merely a methodological preference but a prerequisite for meaningful spatial comparison.

Existing research has established that accident risk is spatially structured, driven by socio-economic deprivation, road network characteristics, and environmental conditions (Jones *et al.*, 2008; Abdel-Aty and Radwan, 2000). What is less well understood is the extent to which these relationships vary across space. Standard regression approaches assume that predictor effects are spatially constant, yet this assumption is increasingly difficult to defend at a national scale, where urban, suburban, and rural contexts generate markedly different risk processes (Tang *et al.*, 2023; Jinli *et al.*, 2024).

This study addresses road traffic accident risk across Middle Layer Super Output Areas (MSOA) in England using STATS19 collision records from 2021 to 2024. Three analytical approaches are applied in sequence: spatial autocorrelation analysis to characterise the geographical clustering of accident rates; Multi-scale Geographically Weighted Regression (MGWR) (Fotheringham *et al.*, 2017) to model spatially varying predictor-outcome relationships; and a Random Forest classifier, trained with spatial block cross-validation (Roberts *et al.*, 2017) and SMOTE, to identify high-risk areas. The combination is designed to move from pattern description through mechanism explanation to predictive application.


| [1.0 Introduction](#1.0-Introduction) | [2.0 Literature Reviews](#2.0-Literature-Reviews) | [3.0 Methodology](#3.0-Methodology) | [4.0 Data](#4.0-Data) | [5.0 Analysis](#5.0-Analysis) | [6.0 Discussion and Conclusion](#6.0-Discussion-and-Conclusion) | [References](#References) |

<a id="Literature-Reviews"></a>
## 2.0 Literature Reviews ##


The relationship between socio-economic deprivation and road casualty risk has been well established across multiple English spatial contexts. Jones *et al.* (2008) found that residents in the most deprived quintile faced substantially elevated road traffic mortality relative to the least deprived, even after controlling for exposure. More recent official statistics confirm that this gradient has not narrowed: between 2020 and 2024, pedestrian casualty rates in the most deprived decile of English areas remained nearly three times those in the least deprived decile (Department for Transport, 2025). Liu *et al.* (2024) analysing crash frequency at the census block group level in the United States, argue that such disparities are poorly captured by global regression models, precisely because the strength and direction of the deprivation-risk relationship varies by local context. 

At the same time, road network structure exerts an independent influence on where collisions occur. Abdel-Aty and Radwan (2000) established that junction density is among the strongest predictors of accident frequency, as intersections concentrate vehicle-vehicle and vehicle-pedestrian conflict. Tang *et al.* (2023) extended this to a multi-scale spatial context, showing that infrastructure variables predict crash occurrence at a finer spatial scale than socio-economic variables - a finding replicated here (IMD bandwidth = 55 MSOAs versus road density bandwidth = 2,807 MSOAs in the MGWR results). Environmental conditions, particularly darkness and wet road surfaces, further amplify risk at the area level; aggregating these collision-level attributes to MSOAs as proportional variables provides a practical proxy for both infrastructure quality and local weather exposure. 

From a methodological standpoint, the inadequacy of spatially homogeneous models for road safety analysis has motivated a growing body of geographically weighted approaches. Brunsdon *et al.* (1996) introduced Geographically Weighted Regression as a way to relax the stationarity assumption, but standard GWR imposes a single bandwidth across all variables, which conflates processes operating at different spatial scales. MGWR addresses this limitation by estimating variable-specific bandwidths (Fotheringham *et al.*, 2017), and Liu *et al.* (2024) confirm that MGWR outperforms both OLS and GWR when the goal is to decompose spatially varying predictor effects in crash frequency analysis.

On the machine learning side, ensemble tree methods have been applied to road safety classification at various spatial scales. Wang *et al.* (2024) demonstrate through a Geographically Weighted Random Forest that the importance of intersection density in predicting crash frequency varies considerably across US countries, and that global Random Forest models systematically understate this spatial heterogeneity. A related concern is evaluation design: Roberts *et al.* (2017) show that conventional k-fold cross-validation produces optimistically biased performance estimates when training and test observations are spatially proximate, and recommend spatial block cross-validation as a more realistic alternative. Both issues directly inform the modelling choices made in this study.



| [1.0 Introduction](#1.0-Introduction) | [2.0 Literature Reviews](#2.0-Literature-Reviews) | [3.0 Methodology](#3.0-Methodology) | [4.0 Data](#4.0-Data) | [5.0 Analysis](#5.0-Analysis) | [6.0 Discussion and Conclusion](#6.0-Discussion-and-Conclusion) | [References](#References) |

<a id="Methodology"></a>
## 3.0 Methodology ##

**Spatial autocorrelation**

A Queen contiguity spatial weights matrix is constructed, defining neighbours as MSOAs sharing any boundary or vertex. Global Moran's I (Moran, 1950) is computed to test for overall spatial clustering, followed by Local Indicators of Spatial Association (LISA; Anselin, 1995) to identify statistically significant local clusters:

- HH (High-High): Hot spots — high-rate MSOAs surrounded by high-rate neighbours
- LL (Low-Low): Cold spots — low-rate MSOAs surrounded by low-rate neighbours
- HL / LH: Spatial outliers

Significance is assessed using 999 random permutations with α = 0.05.

**MGWR**

An OLS regression is first fitted as a baseline:

$$\log(\text{accident\_rate}) = \beta_0 + \beta_1 \cdot \text{imd} + \beta_2 \cdot \text{junction\_density} + \beta_3 \cdot \text{road\_density} + \beta_4 \cdot \text{urban\_pct} + \beta_5 \cdot \text{dark\_pct} + \beta_6 \cdot \text{wet\_road\_pct} + \varepsilon$$

Moran's I on OLS residuals tests for spatial dependence. MGWR (Fotheringham *et al.*, 2017) then allows each coefficient to vary spatially with variable-specific bandwidths, capturing multi-scale processes. Bandwidth selection uses AICc minimisation with an adaptive bi-square kernel.

**Machine learning classification**

MSOAs in the top 20th percentile of accident rate are labelled as "high risk". A Random Forest classifier (Breiman, 2001) is trained with:
- SMOTE (Chawla *et al.*, 2002) to address the 4:1 class imbalance
- Spatial block cross-validation (Roberts *et al.*, 2017) to prevent spatial data leakage

Performance is evaluated using the Precision-Recall curve and Average Precision (more informative than ROC for imbalanced problems).

Five of the six independent variables exhibit VIF values above 5 (Table 2), reflecting the collinearity expected when road network density, junction density, and urban classification are derived from the same spatial context. This does not invalidate the MGWR analysis: because each local regression is estimated over a restricted spatial neighbourhood rather than the full dataset, the effective collinearity is reduced relative to OLS, and the variable-specific bandwidth selection further guards against instability in coefficient estimates (Fotheringham *et al.*, 2017). Random Forest is non-parametric and entirely insensitive to multicollinearity. The OLS model is retained only as a baseline for model comparison and residual autocorrelation testing, and its coefficients are not used for inferential purposes.


| [1.0 Introduction](#1.0-Introduction) | [2.0 Literature Reviews](#2.0-Literature-Reviews) | [3.0 Methodology](#3.0-Methodology) | [4.0 Data](#4.0-Data) | [5.0 Analysis](#5.0-Analysis) | [6.0 Discussion and Conclusion](#6.0-Discussion-and-Conclusion) | [References](#References) |

<a id="Data"></a>
## 4.0 Data ##

This study integrates five datasets at the MSOA level for England. The spatial unit of analysis is the **Middle Layer Super Output Area (MSOA) 2021**, of which there are approximately 6,791 in England, each containing a mean population of ~7,200 residents.


| [1.0 Introduction](#1.0-Introduction) | [2.0 Literature Reviews](#2.0-Literature-Reviews) | [3.0 Methodology](#3.0-Methodology) | [4.0 Data](#4.0-Data) | [5.0 Analysis](#5.0-Analysis) | [6.0 Discussion and Conclusion](#6.0-Discussion-and-Conclusion) | [References](#References) |

### 4.1 Data sources ###

**Table 1**

| Dataset | Source | Temporal Coverage | Spatial Resolution | Format |
|---------|--------|-------------------|-------------------|--------|
| STATS19 Collision Records | Department for Transport | 2021-2024 | Point (Easting/Northing) | CSV |
| MSOA 2021 Boundaries | ONS Open Geography Portal | 2021 | Polygon | GeoPackage |
| IMD 2019 Deprivation Index | MHCLG | 2019 | LSOA (aggregated to MSOA) | CSV |
| OS Open Roads | Ordnance Survey | Current | Line network | GeoPackage |
| Mid-year Population Estimates | ONS (Nomis) | Latest | MSOA | CSV |

### 4.2 Variable description ###

**Table 2**
| Variable | Type | Description | Notes |
|----------|------|-------------|-------|
| `accident_rate_per_10k` | Numeric (continuous) | Collision count per 10,000 population (2021-2024 combined). Dependent variable in regression. | Log-transformed for MGWR |
| `imd_score` | Numeric (continuous) | Population-weighted mean IMD 2019 score per MSOA. Higher = more deprived. | Aggregated from LSOA via lookup table |
| `road_density` | Numeric (continuous) | Total road length (km) per sq km of MSOA area. | Derived from OS Open Roads |
| `junction_density` | Numeric (continuous) | Number of road junctions per sq km. | Derived from OS Open Roads |
| `urban_pct` | Numeric (proportion) | Proportion of collisions classified as urban within the MSOA. | From STATS19 `urban_or_rural_area` field |
| `wet_road_pct` | Numeric (proportion) | Proportion of collisions on wet/damp/flooded road surfaces. | From STATS19 `road_surface_conditions` field |
| `dark_pct` | Numeric (proportion) | Proportion of collisions occurring in darkness (with or without street lighting). | From STATS19 `light_conditions` field |
| `high_risk` | Binary (0/1) | 1 if MSOA is in top 20th percentile of accident rate; 0 otherwise. Target for ML classification. | Threshold at P80 |

### 4.3 Environment setup and data acquisition

In [ ]:
import sys, os, subprocess, warnings
from pathlib import Path

# Installing missing packages in SDS Docker
subprocess.run([
    sys.executable, '-m', 'pip', 'install',
    'loguru==0.7.3', 
    'mgwr==2.2.1', 
    'watermark==2.6.0', 
    'imbalanced-learn>=0.11.0',
    'esda>=2.5.0',
    '-q'
])

import pandas as pd
import geopandas as gpd
import numpy as np
import shutil
import mgwr
import tqdm
import sklearn
import imblearn
import loguru
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from esda.moran import Moran
from loguru import logger
from scipy.interpolate import LinearNDInterpolator

warnings.filterwarnings("ignore", message="The weights matrix is not fully connected")
logger.remove()
logger.add(sys.stderr, level="ERROR")

# Enable automatic module reloading in Jupyter
%load_ext autoreload
%autoreload 2

# Data link
REPO_URL = "https://github.com/sakura-lee25/Data-Science-for-Spatial-Systems_final-assessment.git"
_project_root = Path.home() / "uk-traffic-project"

if _project_root.is_dir():
    print("Detected existing local repository. Removing to ensure a fresh pull...")
    shutil.rmtree(_project_root)
   
print("Cloning latest repository from GitHub (includes data + source code) ...")
subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(_project_root)],
    check=True,
    )
print("Clone complete.")

sys.path.insert(0, str(_project_root))
os.chdir(_project_root)

from src.utils.config import (
    ensure_dirs, ensure_file, RAW_DIR, PROCESSED_DIR, FIGURES_DIR,
    MSOA_ANALYSIS_GPKG, MGWR_COEFFICIENTS_GPKG,
    YEARS, FIGURE_DPI,
)

plt.rcParams.update({
    'font.family': 'serif',
    'figure.dpi': 150,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'figure.facecolor': 'white'
})

ensure_dirs()
print("Environment ready.")

In [ ]:
# Load datasets from cloned repository
# All raw data and pre-computed results are included in the repo.

stats19_frames = {
    y: pd.read_csv(
    RAW_DIR / f"stats19_collision_{y}.csv",
    dtype={0: str, 2: str},   # accident_index, accident_reference
    low_memory=False
) for y in YEARS
}
lookup_df = pd.read_csv(RAW_DIR / "lsoa_msoa_lookup.csv")
imd_df    = pd.read_csv(RAW_DIR / "imd_2019_scores.csv")
pop_df    = pd.read_csv(RAW_DIR / "msoa_population.csv")

print("DATA LOADED")
for y, df in stats19_frames.items():
    print(f"  STATS19 {y}: {len(df):>8,} rows, {len(df.columns)} cols")
print(f"  LSOA-MSOA lookup:     {len(lookup_df):>8,} rows")
print(f"  IMD 2019:             {len(imd_df):>8,} rows")
print(f"  Population:           {len(pop_df):>8,} rows")

### 4.4 Raw data preview

A sample of the STATS19 collision records:

In [ ]:
latest_year = max(YEARS)
df_preview = stats19_frames[latest_year]
print(f"STATS19 {latest_year} — {len(df_preview):,} records, {len(df_preview.columns)} columns")
print(f"Columns: {list(df_preview.columns)[:10]}...")
df_preview.head(3)

In [ ]:
msoa_local = ensure_file(RAW_DIR / "msoa_2021_boundaries.gpkg")
msoa_gdf = gpd.read_file(msoa_local)
print(f"MSOA boundaries: {msoa_gdf.shape[0]:,} polygons, CRS = {msoa_gdf.crs}")
msoa_gdf.head(3)

### 4.5 Data preprocessing and feature engineering

The preprocessing pipeline:
1. Cleans STATS19 records and creates binary indicator variables
2. Reprojects accident points to OSGB36 (EPSG:27700) and spatially joins to MSOA polygons
3. Aggregates IMD 2019 scores from LSOA to MSOA using population-weighted means
4. Computes road network metrics (road density, junction density) per MSOA
5. Merges all features and computes `accident_rate_per_10k`

In [ ]:
from src.analysis.preprocessing.feature_builder import build_features

gdf = build_features(force=False)
print(f"Feature dataset shape: {gdf.shape}")
print(f"CRS: {gdf.crs}")
print(f"Memory usage: {gdf.memory_usage(deep=True).sum() / 1e6:.1f} MB")

In [ ]:
gdf.columns = gdf.columns.str.lower()

print(f"Number of MSOAs: {len(gdf):,}")
print(f"\nColumn types:")
for col in gdf.columns:
    if col == 'geometry':
        continue
    dtype = str(gdf[col].dtype)
    null_count = gdf[col].isnull().sum()
    null_pct = f"({null_count / len(gdf) * 100:.1f}% missing)" if null_count > 0 else ""
    print(f"  {col:30s} {dtype:15s} {null_pct}")

In [ ]:
gdf.describe()

In [ ]:
gdf.head()

**Figure 1** | Distribution of key analysis variables across MSOAs

In [ ]:
plot_cols = [
    'accident_rate_per_10k', 'imd_score', 'road_density',
    'junction_density', 'urban_pct', 'wet_road_pct', 'dark_pct'
]
available_cols = [c for c in plot_cols if c in gdf.columns]

n_cols = 4
n_rows = int(np.ceil(len(available_cols) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows))
axes_flat = axes.flatten()

for i, col in enumerate(available_cols):
    gdf[col].hist(bins=50, ax=axes_flat[i], edgecolor='white', alpha=0.8)
    axes_flat[i].set_title(col.replace('_', ' ').title(), fontsize=11)
    axes_flat[i].set_ylabel('Frequency')

for j in range(len(available_cols), len(axes_flat)):
    axes_flat[j].set_visible(False)

plt.suptitle('Figure 1: Distribution of Key Variables across MSOAs',
             fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig01_variable_distributions.png', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()

**Figure 2** | Correlation matrix of MSOA features

In [ ]:
numeric_cols = [c for c in gdf.select_dtypes(include=[np.number]).columns
                if c not in ['objectid', 'OBJECTID']]
corr = gdf[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax, square=True,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
ax.set_title('Figure 2: Correlation Matrix of MSOA Features', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig02_correlation_matrix.png', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()

**Figure 3** | Spatial distribution of accident rate and deprivation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 10))

gdf.plot(column='accident_rate_per_10k', cmap='YlOrRd', legend=True, ax=axes[0],
         edgecolor='face', linewidth=0.1,
         legend_kwds={'label': 'Rate per 10,000', 'shrink': 0.6})
axes[0].set_title('(a) Accident Rate per 10,000 Population', fontweight='bold')
axes[0].axis('off')

gdf.plot(column='imd_score', cmap='RdPu', legend=True, ax=axes[1],
         edgecolor='face', linewidth=0.1,
         legend_kwds={'label': 'IMD Score', 'shrink': 0.6})
axes[1].set_title('(b) IMD 2019 Deprivation Score', fontweight='bold')
axes[1].axis('off')

plt.suptitle('Figure 3: Spatial Distribution of Accident Rate and Deprivation',
             fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig03_choropleth_rate_imd.png', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()

<a id="Analysis"></a>
## 5.0 Analysis


The analytical framework comprises three complementary approaches:

1. **Spatial Autocorrelation Analysis** — Global and Local Moran's I to detect and map spatial clustering of accident rates.

2. **Multi-scale Geographically Weighted Regression (MGWR)** — to model spatially varying relationships between accident rates and explanatory variables at multiple spatial scales.

3. **Random Forest Classification** — to predict high-risk MSOAs using socio-economic and infrastructure features, with SMOTE for class imbalance and spatial block cross-validation.

All analyses are conducted at the MSOA level using EPSG:27700 (OSGB36) as the coordinate reference system. The Isles of Scilly (E02006781) were excluded prior to weight construction as a geographic island with no contiguous neighbours under the Queen criterion.

| [1.0 Introduction](#1.0-Introduction) | [2.0 Literature Reviews](#2.0-Literature-Reviews) | [3.0 Methodology](#3.0-Methodology) | [4.0 Data](#4.0-Data) | [5.0 Analysis](#5.0-Analysis) | [6.0 Discussion and Conclusion](#6.0-Discussion-and-Conclusion) | [References](#References) |

In [ ]:
# Methodology flowchart
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(0, 14)
ax.set_ylim(0, 8)
ax.axis('off')

def draw_box(ax, x, y, w, h, text, color='#EBF5FB', edge='#2980B9', fontsize=9):
    box = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.15",
                          facecolor=color, edgecolor=edge, linewidth=1.5)
    ax.add_patch(box)
    ax.text(x + w/2, y + h/2, text, ha='center', va='center',
            fontsize=fontsize, fontweight='bold', wrap=True)

def draw_arrow(ax, x1, y1, x2, y2):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='#2C3E50', lw=1.8))

# Column 1: Data Sources
draw_box(ax, 0.3, 6.5, 3, 1, 'STATS19\nCollision Records\n(2021-2024)', '#FADBD8', '#E74C3C')
draw_box(ax, 0.3, 5.0, 3, 1, 'MSOA 2021\nBoundaries (ONS)', '#FADBD8', '#E74C3C')
draw_box(ax, 0.3, 3.5, 3, 1, 'IMD 2019\nDeprivation Index', '#FADBD8', '#E74C3C')
draw_box(ax, 0.3, 2.0, 3, 1, 'OS Open Roads\nNetwork', '#FADBD8', '#E74C3C')
draw_box(ax, 0.3, 0.5, 3, 1, 'ONS Population\nEstimates', '#FADBD8', '#E74C3C')

# Column 2: Preprocessing
draw_box(ax, 4.5, 4.0, 3, 2.5, 'Preprocessing\n─────────\nClean & geocode\nSpatial join\nLSOA→MSOA agg\nRoad metrics\nFeature merge', '#D5F5E3', '#27AE60')

# Arrows: sources → preprocessing
for y_start in [7.0, 5.5, 4.0, 2.5, 1.0]:
    draw_arrow(ax, 3.3, y_start, 4.5, 5.25)

# Column 3: Analysis methods
draw_box(ax, 8.5, 6.0, 3.5, 1.2, 'RQ1: Spatial\nAutocorrelation\n(Moran\'s I + LISA)', '#EBF5FB', '#2980B9')
draw_box(ax, 8.5, 4.0, 3.5, 1.2, 'RQ2: MGWR\nMulti-scale\nRegression', '#EBF5FB', '#2980B9')
draw_box(ax, 8.5, 2.0, 3.5, 1.2, 'RQ3: Random Forest\nClassification\n(SMOTE + Spatial CV)', '#EBF5FB', '#2980B9')

# Arrow: preprocessing → analyses
draw_arrow(ax, 7.5, 5.8, 8.5, 6.6)
draw_arrow(ax, 7.5, 5.0, 8.5, 4.6)
draw_arrow(ax, 7.5, 4.2, 8.5, 2.6)

# Title
ax.text(7, 7.6, 'Figure 4: Analytical Framework', ha='center', fontsize=13,
        fontweight='bold', style='italic')

# Column labels
ax.text(1.8, 7.8, 'Data Sources', ha='center', fontsize=11, fontweight='bold', color='#E74C3C')
ax.text(6.0, 7.8, 'Processing', ha='center', fontsize=11, fontweight='bold', color='#27AE60')
ax.text(10.25, 7.8, 'Analysis', ha='center', fontsize=11, fontweight='bold', color='#2980B9')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig04_analytical_flowchart.png', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()

### 5.1 Spatial autocorrelation of accident rates

In [ ]:
# Filter to positive-rate MSOAs for spatial analysis
gdf_analysis = gdf[gdf['accident_rate_per_10k'] > 0].copy()

# Exclude Isles of Scilly (E02006781) — geographic island with no contiguous neighbours
gdf_analysis = gdf_analysis[gdf_analysis['msoa_code'] != 'E02006781'].reset_index(drop=True)

print(f"MSOAs for spatial analysis: {len(gdf_analysis):,} (after removing zero-rate areas and island outlier)")

In [ ]:
from src.analysis.spatial_autocorr import build_spatial_weights, global_morans_i

# Constructing a spatial weight matrix
w = build_spatial_weights(gdf_analysis)

# Calculating global_morans_i
global_result = global_morans_i(gdf_analysis, variable='accident_rate_per_10k', w=w)

print("Global Moran's I Test")
print(f"  Moran's I:   {global_result['I']:.4f}")
print(f"  Expected I:  {global_result['expected_I']:.4f}")
print(f"  Z-score:     {global_result['z_score']:.4f}")
print(f"  p-value:     {global_result['p_value']:.6f}")

if global_result['p_value'] < 0.001:
    print("\n→ Highly significant positive spatial autocorrelation (p < 0.001).")
    print("  Neighbouring MSOAs tend to have similar accident rates.")

**Figure 5** | Moran scatter plot

In [ ]:
y = gdf_analysis['accident_rate_per_10k'].values
mi = Moran(y, w, permutations=999)

y_std = (y - y.mean()) / y.std()
wy_std = np.array([sum(w[i][j] * y_std[j] for j in w[i]) for i in range(len(y_std))])

fig, ax = plt.subplots(figsize=(8, 7))
ax.scatter(y_std, wy_std, s=4, alpha=0.3, color='steelblue')

m, b = np.polyfit(y_std, wy_std, 1)
x_line = np.linspace(y_std.min(), y_std.max(), 100)
ax.plot(x_line, m * x_line + b, color='red', linewidth=2, label=f"slope = {m:.4f}")

ax.axhline(0, color='grey', ls='--', lw=0.8)
ax.axvline(0, color='grey', ls='--', lw=0.8)

ax.text(0.95, 0.95, 'HH', transform=ax.transAxes, fontsize=14, fontweight='bold',
        ha='right', va='top', color='red', alpha=0.5)
ax.text(0.05, 0.05, 'LL', transform=ax.transAxes, fontsize=14, fontweight='bold',
        ha='left', va='bottom', color='blue', alpha=0.5)
ax.text(0.95, 0.05, 'HL', transform=ax.transAxes, fontsize=14, fontweight='bold',
        ha='right', va='bottom', color='orange', alpha=0.5)
ax.text(0.05, 0.95, 'LH', transform=ax.transAxes, fontsize=14, fontweight='bold',
        ha='left', va='top', color='lightblue', alpha=0.5)

ax.set_xlabel('Standardised Accident Rate (z)', fontsize=12)
ax.set_ylabel('Spatial Lag of Accident Rate (Wz)', fontsize=12)
ax.set_title(f"Figure 5: Moran Scatter Plot (I = {mi.I:.4f}, p = {mi.p_sim:.4f})",
             fontweight='bold', fontsize=13)
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig05_moran_scatter.png', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()

In [ ]:
from src.analysis.spatial_autocorr import local_morans_i

lisa_gdf = local_morans_i(gdf_analysis, variable='accident_rate_per_10k', w=w)

# Fixing random seeds
np.random.seed()

print("LISA Cluster Distribution")
cluster_counts = lisa_gdf['lisa_cluster'].value_counts()
for cluster, count in cluster_counts.items():
    pct = count / len(lisa_gdf) * 100
    print(f"  {cluster:5s}  {count:5,d}  ({pct:.1f}%)")

**Figure 6** | LISA cluster map — accident rate

In [ ]:
COLORS = {'HH': '#d7191c', 'LL': '#2c7bb6', 'HL': '#fdae61', 'LH': '#abd9e9', 'NS': '#d3d3d3'}
LABELS = {
    'HH': 'High-High (Hot Spot)',
    'LL': 'Low-Low (Cold Spot)',
    'HL': 'High-Low (Outlier)',
    'LH': 'Low-High (Outlier)',
    'NS': 'Not Significant'
}

fig, ax = plt.subplots(figsize=(10, 12))
patches = []
for cluster, colour in COLORS.items():
    subset = lisa_gdf[lisa_gdf['lisa_cluster'] == cluster]
    if not subset.empty:
        subset.plot(ax=ax, color=colour, edgecolor='face', linewidth=0.1)
        patches.append(mpatches.Patch(color=colour, label=LABELS[cluster]))

ax.legend(handles=patches, loc='lower left', fontsize=10, framealpha=0.9)
ax.set_title('Figure 6: LISA Clusters — Accident Rate per 10,000 (2021-2024)',
             fontweight='bold', fontsize=13)
ax.axis('off')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig06_lisa_clusters.png', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()

#### 5.1.1 Yearly sensitivity analysis

To test the temporal robustness of spatial patterns, Global Moran's I is computed for each year separately. Note that 2021 is the first post-COVID recovery year.

In [ ]:
from src.analysis.spatial_autocorr import sensitivity_analysis

ISLAND_CODES = ['E02006781', 'E02006741', 'E02006742', 'E02006752']
gdf = gdf[~gdf['msoa_code'].isin(ISLAND_CODES)].reset_index(drop=True)

yearly_df = sensitivity_analysis()

print("Table 3: Yearly Global Moran's I")
print("═" * 55)
print(f"{'Year':>6s}  {'Moran I':>10s}  {'Z-score':>10s}  {'p-value':>10s}  {'Sig.':>6s}")
print("─" * 55)
for _, row in yearly_df.iterrows():
    sig = '***' if row['p_value'] < 0.001 else '**' if row['p_value'] < 0.01 else '*' if row['p_value'] < 0.05 else 'ns'
    print(f"{int(row['year']):>6d}  {row['I']:>10.4f}  {row['z_score']:>10.4f}  {row['p_value']:>10.6f}  {sig:>6s}")
print("═" * 55)
print("Significance: *** p < 0.001, ** p < 0.01, * p < 0.05")

**Figure 7** | LISA clusters by year (2021-2024)

In [ ]:
import warnings

yearly_lisa = {}
for year in YEARS:
    path = ensure_file(PROCESSED_DIR / f"msoa_yearly_{year}.gpkg")
    if path.exists():
        y_gdf = gpd.read_file(path)
        y_gdf = y_gdf[y_gdf['accident_rate_per_10k'] > 0].copy()

        # Excluding known isolated islands (MSOA)
        island_codes = ['E02006781', 'E02006741', 'E02006742', 'E02006752']
        y_gdf = y_gdf[~y_gdf['msoa_code'].isin(island_codes)].reset_index(drop=True)

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")   # Block RuntimeWarning
            yearly_lisa[year] = local_morans_i(y_gdf, variable='accident_rate_per_10k')


if yearly_lisa:
    fig, axes = plt.subplots(1, len(yearly_lisa), figsize=(6 * len(yearly_lisa), 8))
    if len(yearly_lisa) == 1:
        axes = [axes]

    for idx, (year, lgdf) in enumerate(sorted(yearly_lisa.items())):
        ax = axes[idx]
        for cluster, colour in COLORS.items():
            subset = lgdf[lgdf['lisa_cluster'] == cluster]
            if not subset.empty:
                subset.plot(ax=ax, color=colour, edgecolor='face', linewidth=0.1)
        note = ' (post-COVID)' if year == 2021 else ''
        ax.set_title(f'{year}{note}', fontweight='bold', fontsize=12)
        ax.axis('off')

    plt.suptitle('Figure 7: LISA Clusters by Year', fontweight='bold', fontsize=14)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'fig07_lisa_yearly.png', dpi=FIGURE_DPI, bbox_inches='tight')
    plt.show()


### 5.2 Spatially Varying Relationships

In [ ]:
# Override module variables to use full 6-variable specification
import src.analysis.mgwr_analysis as mgwr_mod
import src.analysis.ml_classification as ml_mod

INDEPENDENT_VARS = [
    'imd_score', 'junction_density', 'road_density',
    'urban_pct', 'dark_pct', 'wet_road_pct',
]
mgwr_mod.INDEPENDENT_VARS = INDEPENDENT_VARS
mgwr_mod.DEPENDENT_VAR = 'log_accident_rate'

ml_mod.FEATURE_COLS = [
    'imd_score', 'junction_density', 'road_density', 'urban_pct',
    'dark_pct', 'wet_road_pct', 'road_length_km', 'population',
]

from src.analysis.mgwr_analysis import check_vif

vif_df = check_vif(gdf_analysis, variables=INDEPENDENT_VARS)

print("Table 4: VIF Multicollinearity Check")
print("═" * 45)
for _, row in vif_df.iterrows():
    flag = '  Warning' if row['VIF'] > 5 else '  OK'
    print(f"  {row['variable']:25s} VIF = {row['VIF']:6.2f}{flag}")
print("═" * 45)

vif_df


In [ ]:
from src.analysis.mgwr_analysis import run_ols_baseline

ols = run_ols_baseline(gdf_analysis)

print("Table 5: OLS Regression Results")
print("═" * 60)
print(f"  R²:       {ols['r_squared']:.4f}")
print(f"  Adj R²:   {ols['adj_r_squared']:.4f}")
print(f"  AIC:      {ols['aic']:.1f}")
print(f"")
print(f"  Residual Moran's I: {ols['residual_moran_I']:.4f} (p = {ols['residual_moran_p']:.6f})")
print("═" * 60)

print(f"\n{'Variable':>25s}  {'Coefficient':>12s}  {'Sig.':>5s}")
print("─" * 47)
for var, coef in ols['coefficients'].items():
    pval = ols['p_values'][var]
    sig = '***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else 'ns'
    print(f"  {var:>25s}  {coef:>12.4f}  {sig:>5s}")
print("─" * 47)

if ols['residual_moran_p'] < 0.05:
    print(f"\n→ Significant spatial autocorrelation in OLS residuals.")
    print("  This justifies the use of GWR/MGWR.")


In [ ]:
from src.utils.config import MGWR_COEFFICIENTS_GPKG, ensure_file

# Load pre-computed MGWR results from coefficient GeoPackage
coef_path = ensure_file(MGWR_COEFFICIENTS_GPKG)
coef_gdf = gpd.read_file(coef_path)

# Pre-computed model metrics from validated execution run
mgwr_results = {
    'gwr_r2': 0.4535,
    'gwr_aicc': 9585.5,
    'mgwr_r2': 0.4575,
    'mgwr_aicc': 9448.7,
    'mgwr_bw': [55, 1740, 2807, 6676, 727, 500],
    'variable_names': ['intercept'] + INDEPENDENT_VARS,
    'coef_gdf': coef_gdf,
    'n_obs': len(coef_gdf),
}

print("Table 6: Model Comparison")
print("═" * 50)
print(f"{'Model':>8s}  {'R²':>8s}  {'AICc':>12s}")
print("─" * 50)
print(f"{'OLS':>8s}  {ols['r_squared']:>8.4f}  {ols['aic']:>12.1f}")
print(f"{'GWR':>8s}  {mgwr_results['gwr_r2']:>8.4f}  {mgwr_results['gwr_aicc']:>12.1f}")
print(f"{'MGWR':>8s}  {mgwr_results['mgwr_r2']:>8.4f}  {mgwr_results['mgwr_aicc']:>12.1f}")
print("═" * 50)

comparison = pd.DataFrame({
    'Model': ['OLS', 'GWR', 'MGWR'],
    'R²': [ols['r_squared'], mgwr_results['gwr_r2'], mgwr_results['mgwr_r2']],
    'AICc': [ols['aic'], mgwr_results['gwr_aicc'], mgwr_results['mgwr_aicc']],
})
comparison


In [ ]:
print("Table 7: MGWR Variable Bandwidths")
print("═" * 50)
print(f"{'Variable':>25s}  {'Bandwidth':>12s}  {'Scale':>10s}")
print("─" * 50)

n_obs = len(gdf_analysis)
var_names = mgwr_results['variable_names']
bws = ['global'] + list(mgwr_results['mgwr_bw'])

for var, bw in zip(var_names, bws):
    if isinstance(bw, (int, float)):
        scale = 'Local' if bw < n_obs * 0.3 else 'Regional' if bw < n_obs * 0.7 else 'Global'
        print(f"  {var:>25s}  {bw:>12.0f}  {scale:>10s}")
    else:
        print(f"  {var:>25s}  {str(bw):>12s}  {'Global':>10s}")
print("═" * 50)


In [ ]:
coef_gdf = mgwr_results['coef_gdf']
print(f"MGWR coefficient surface: {coef_gdf.shape}")
coef_gdf[INDEPENDENT_VARS].describe()

**Figure 8** | MGWR local coefficient surfaces

In [ ]:
n_vars = len(INDEPENDENT_VARS)
n_cols = 3
n_rows = int(np.ceil(n_vars / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 7 * n_rows))
axes_flat = axes.flatten()

for i, var in enumerate(INDEPENDENT_VARS):
    ax = axes_flat[i]
    vmax = max(abs(coef_gdf[var].quantile(0.02)), abs(coef_gdf[var].quantile(0.98)))
    coef_gdf.plot(column=var, cmap='RdBu_r', vmin=-vmax, vmax=vmax,
                  legend=True, ax=ax, edgecolor='face', linewidth=0.1,
                  legend_kwds={'shrink': 0.6})
    ax.set_title(var.replace('_', ' ').title(), fontweight='bold', fontsize=11)
    ax.axis('off')

for j in range(n_vars, len(axes_flat)):
    axes_flat[j].set_visible(False)

plt.suptitle('Figure 8: MGWR Local Coefficients', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig08_mgwr_coefficients.png', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()

### 5.3 Machine learning classification

In [ ]:
from src.analysis.ml_classification import create_labels, FEATURE_COLS

gdf_ml = gdf_analysis.copy()
gdf_ml = create_labels(gdf_ml)

print("Class Distribution:")
class_counts = gdf_ml['high_risk'].value_counts()
for label, count in class_counts.items():
    pct = count / len(gdf_ml) * 100
    name = 'High Risk' if label == 1 else 'Low Risk'
    print(f"  {name:12s} (label={label}):  {count:5,d}  ({pct:.1f}%)")
print(f"\n  Imbalance ratio: 1:{class_counts[0] / class_counts[1]:.1f}")

In [ ]:
from src.analysis.ml_classification import train_random_forest

results = train_random_forest(gdf_ml, spatial_cv=True, use_smote=True)

print("Table 8: Random Forest Results (Spatial Block CV + SMOTE)")
print("═" * 50)
print(f"  Mean CV F1-score:      {results['mean_f1']:.3f}")
print(f"  Average Precision:     {results['avg_precision']:.3f}")
print(f"")
print(f"  Per-fold F1 scores:")
for i, score in enumerate(results['fold_f1_scores']):
    print(f"    Fold {i+1}: {score:.3f}")
print("═" * 50)

**Figure 9** | Precision-Recall curve

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(results['recall'], results['precision'], linewidth=2, color='#e74c3c')
ax.fill_between(results['recall'], results['precision'], alpha=0.1, color='#e74c3c')

baseline = sum(gdf_ml['high_risk'] == 1) / len(gdf_ml)
ax.axhline(baseline, color='grey', linestyle='--', linewidth=1,
           label=f'Baseline = {baseline:.2f}')

ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title(f"Figure 9: Precision-Recall Curve (AP = {results['avg_precision']:.3f})",
             fontweight='bold', fontsize=13)
ax.legend(fontsize=11)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig09_pr_curve.png', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()

**Figure 10** | Confusion matrix

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(results['confusion_matrix'], annot=True, fmt='d', cmap='Blues',
            xticklabels=['Low Risk', 'High Risk'],
            yticklabels=['Low Risk', 'High Risk'], ax=ax,
            annot_kws={'fontsize': 14})
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Actual', fontsize=12)
ax.set_title('Figure 10: Confusion Matrix', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig10_confusion_matrix.png', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()

In [ ]:
from sklearn.metrics import classification_report

print("Table 9: Classification Report")
print("═" * 55)
print(classification_report(
    results['y_true'], results['y_pred'],
    target_names=['Low Risk', 'High Risk'],
    digits=3
))
print("═" * 55)

**Figure 11** | Feature importance

In [ ]:
fi = results['feature_importance'].sort_values('importance', ascending=False)

print("Feature Importance Ranking")
print("═" * 45)
for rank, (_, row) in enumerate(fi.iterrows(), 1):
    bar = '█' * int(row['importance'] * 50)
    print(f"  {rank}. {row['feature']:25s} {row['importance']:.4f}  {bar}")
print("═" * 45)

# Horizontal bar chart
fig, ax = plt.subplots(figsize=(10, 6))
fi_sorted = fi.sort_values('importance', ascending=True)
ax.barh(fi_sorted['feature'].str.replace('_', ' ').str.title(),
        fi_sorted['importance'], color='#3498db', edgecolor='white')
for i, v in enumerate(fi_sorted['importance']):
    ax.text(v + 0.005, i, f'{v:.3f}', va='center', fontweight='bold')
ax.set_xlabel('Mean Decrease in Impurity', fontsize=12)
ax.set_title('Figure 11: Random Forest Feature Importance', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig11_feature_importance.png', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()

**Figure 12** | Actual vs predicted high-risk MSOAs

In [ ]:
available_features = [f for f in FEATURE_COLS if f in gdf_ml.columns]
mask = gdf_ml[available_features + ['high_risk']].dropna().index
gdf_pred = gdf_ml.loc[mask].copy()
gdf_pred['predicted_risk'] = results['y_pred']
gdf_pred['risk_probability'] = results['y_pred_proba']

fig, axes = plt.subplots(1, 2, figsize=(16, 10))

gdf_pred.plot(column='high_risk', cmap='RdYlGn_r', legend=True, ax=axes[0],
              edgecolor='face', linewidth=0.1,
              legend_kwds={'label': 'Risk Class', 'shrink': 0.6})
axes[0].set_title('(a) Actual High-Risk MSOAs (Top 20%)', fontweight='bold', fontsize=12)
axes[0].axis('off')

gdf_pred.plot(column='risk_probability', cmap='RdYlGn_r', legend=True, ax=axes[1],
              edgecolor='face', linewidth=0.1,
              legend_kwds={'label': 'Predicted Probability', 'shrink': 0.6})
axes[1].set_title('(b) Predicted Risk Probability', fontweight='bold', fontsize=12)
axes[1].axis('off')

plt.suptitle('Figure 12: Random Forest — Actual vs Predicted Risk',
             fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig12_risk_prediction_map.png', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()

<a id="Discussion-and-conclusion"></a>
## 6.0 Discussion and Conclusion

| [1.0 Introduction](#1.0-Introduction) | [2.0 Literature Reviews](#2.0-Literature-Reviews) | [3.0 Methodology](#3.0-Methodology) | [4.0 Data](#4.0-Data) | [5.0 Analysis](#5.0-Analysis) | [6.0 Discussion and Conclusion](#6.0-Discussion-and-Conclusion) | [References](#References) |


**RQ1**: Do road traffic accident rates exhibit statistically significant spatial autocorrelation across MSOAs in England, and where are the hot spots and cold spots located?

**RQ2**: How do the relationships between socio-economic deprivation, road infrastructure, and accident risk vary across space, and at what spatial scales do these relationships operate?

**RQ3**: Can area-level socio-economic and infrastructure characteristics reliably predict high-risk MSOAs, and which features are most important for classification?

**Discussion**

**RQ1**: The Global Moran's I of 0.300 (z = 39.53, p ≤ 0.001) confirms highly significant positive spatial autocorrelation in MSOA-level accident rates. LISA analysis further identifies 9.8% of MSOAs in High-High clusters, concentrated around London, Lincoln, West Yorkshire, the West Midlands, where high road and junction densities combine with large pedestrian populations to elevate collision probability regardless of per capita normalisation applied. The 7.5% of Low-Low clusters areas are predominantly rural, consistent with lower traffic volumes and fewer conflict points. This pattern persists across all four years studied (2021–2024), indicating structural rather than transient clustering.

**RQ2**: The MGWR results reveals that predictor–outcome relationships are far from spatially constant. The IMD deprivation score operates at a fine local scale (bandwidth ≈ 55 MSOAs), with deprivation amplifying risk most strongly in inner-city areas; road density operates at a regional scale (bandwidth ≈ 2,807 MSOAs), consistent with its role as a proxy for broad urbanisation. The shift from OLS (R² = 0.12) to MGWR (R² = 0.45) demonstrates that stationary regression fundamentally misrepresents these multi-scale processes.

**RQ3**: The Random Forest classifier achieves a mean spatial block CV F1-score of 0.405 and an Average Precision of 0.398 against a naive baseline of 0.20, demonstrating meaningful but moderate predictive power using aggregate area-level features alone. Dark conditions (dark_pct, importance 0.203) and wet road surfaces (wet_road_pct, importance 0.169) are the most discriminative features, ranking above both road network density and deprivation indices, with direct implications for infrastructure-focused intervention.

The study period (2021-2024) includes 2021 as the first post-COVID recovery year, during which traffic volumes and patterns may not be representative. IMD 2019 scores may not capture recent socio-economic changes. The MSOA unit of analysis may mask within-area heterogeneity. OS Open Roads data represents the current network rather than historical configurations.

**Conclusion**

This study demonstrates that road traffic accident risk across English MSOAs is spatially clustered, structurally persistent, and driven by processes that operate at different geographical scales. MGWR reveals that effective intervention requires locally calibrated strategies rather than uniform national policy. The machine learning results point specifically to lighting and road surface quality as the most discriminative predictors at the aggregate level, suggesting targeted infrastructure investment in the hot-spot clusters identified by the LISA analysis. Future work should incorporate traffic flow data to sharpen the exposure denominator, test the framework at the finer LSOA scale, and introduce temporal disaggregation to better characterise the conditions under which lighting and surface effects dominate.

## References

| [1.0 Introduction](#1.0-Introduction) | [2.0 Literature Reviews](#2.0-Literature-Reviews) | [3.0 Methodology](#3.0-Methodology) | [4.0 Data](#4.0-Data) | [5.0 Analysis](#5.0-Analysis) | [6.0 Discussion and Conclusion](#6.0-Discussion-and-Conclusion) | [References](#References) |

Abdel-Aty, M.A. and Radwan, A.E. (2000) 'Modeling traffic accident occurrence and involvement', *Accident Analysis & Prevention*, 32(5), pp. 633-642.

Anselin, L. (1995) 'Local indicators of spatial association — LISA', *Geographical Analysis*, 27(2), pp. 93-115.

Breiman, L. (2001) 'Random forests', *Machine Learning*, 45(1), pp. 5-32.

Brunsdon, C., Fotheringham, A.S. and Charlton, M.E. (1996) 'Geographically weighted regression: a method for exploring spatial nonstationarity', *Geographical Analysis*, 28(4), pp. 281–298.

Chawla, N.V., Bowyer, K.W., Hall, L.O. and Kegelmeyer, W.P. (2002) 'SMOTE: Synthetic minority over-sampling technique', *Journal of Artificial Intelligence Research*, 16, pp. 321-357.

Department for Transport (2024) Reported road casualties in Great Britain: annual report 2023. *London: Department for Transport*.

Department for Transport (2025) Reported Road Casualties Great Britain: Casualties and Deprivation 2024 (England). *London: Department for Transport*. 

Fotheringham, A.S., Yang, W. and Kang, W. (2017) 'Multiscale geographically weighted regression (MGWR)', *Annals of the American Association of Geographers*, 107(6), pp. 1247-1265.

Jones, A.P., Haynes, R., Kennedy, V., Harvey, I.M., Jewell, T. and Lea, D. (2008) 'Geographical variations in mortality and morbidity from road traffic accidents in England and Wales', *Health & Place*, 14(3), pp. 519-535.

Moran, P.A.P. (1950) 'Notes on continuous stochastic phenomena', *Biometrika*, 37(1/2), pp. 17-23.

Wang, S., Gao, K., Zhang, L., Yu, B., and Easa, S. M. (2024) 'Geographically weighted machine learning for modeling spatial heterogeneity in traffic crash frequency and determinants in US', *Accident Analysis & Prevention*, 199, p. 107528.

Roberts, D.R. *et al.* (2017) 'Cross-validation strategies for data with temporal, spatial, hierarchical, or phylogenetic structure', *Ecography*, 40(8), pp. 913-929.

Tang, X., Bi, R. and Wang, Z. (2023) 'Spatial analysis of moving-vehicle crashes and fixed-object crashes based on multi-scale geographically weighted regression', *Accident Analysis & Prevention*, 189, p. 107123. 

Liu, J., Das, S., and Khan, M. N. (2024) 'Decoding the impacts of contributory factors and addressing social disparities in crash frequency analysis', *Accident Analysis & Prevention*, 194, pp. 233–244.